In [ ]:
# 

In [128]:
# --- CELL 1: imports, paths, knobs ---
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors as mcolors
from pycirclize import Circos

# ---- I/O ----
CSV_IN   = Path("/mnt/c/Users/jonan/Documents/1Work/RoseLab/Spatial/CAR_T/Results/master_df_interactions_all.csv")
OUT_DIR  = Path("/mnt/c/Users/jonan/Documents/1Work/RoseLab/Spatial/CAR_T/figures/R01_figs/Circos")
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_full = pd.read_csv(CSV_IN, sep=None, engine="python")

# ------ Data prep -----------------------------
sample_focus = ["RTCyPSCA_2_4", "CyPSCA_1_1",
                "NoTx_2_2", "RTCyPSCA_1_4"]

label_map = {
    "CyPSCA_1_1": "CyPSCA",
    "NoTx_2_2": "No Treatment",
    "RTCyPSCA_1_4": "RTCyPSCA (RT)",
    "RTCyPSCA_2_4": "RTCyPSCA (No RT)",
}

focus = df["mouse"].unique()
df = df_full[df_full['mouse'].isin(sample_focus)].copy()
print(f"Mice being processed {focus}")

# --- lock positions of common ligands/receptors across plots ---
LOCK_COMMON_POSITIONS = True
SHUFFLE_NONCOMMON     = False     # set False to order non-common by size instead of random
RNG_SEED_BASE         = 20251002 # reproducible randomness per mouse

# Compute “common across all selected mice” sets using the FULL df (not per-topN)
mice_all = sorted(df["mouse"].dropna().unique().tolist())
lig_by_mouse = df.groupby("mouse")["ligand"].apply(lambda s: set(s.astype(str)))
rec_by_mouse = df.groupby("mouse")["receptor"].apply(lambda s: set(s.astype(str)))

COMMON_LIGS = set.intersection(*lig_by_mouse) if len(lig_by_mouse) else set()
COMMON_RECS = set.intersection(*rec_by_mouse) if len(rec_by_mouse) else set()

# Canonical, predictable order for the locked ones (edit if you want a custom order)
CANON_LIG_ORDER = sorted(COMMON_LIGS)
CANON_REC_ORDER = sorted(COMMON_RECS)

print(f"[Common sets] ligands={len(CANON_LIG_ORDER)}, receptors={len(CANON_REC_ORDER)}")

Mice being processed ['RTCyPSCA_1_4' 'CyPSCA_1_1' 'NoTx_2_2' 'RTCyPSCA_2_4']
[Common sets] ligands=29, receptors=15


In [129]:
COMMON_LIGS = set.intersection(*lig_by_mouse) if len(lig_by_mouse) else set()
COMMON_RECS = set.intersection(*rec_by_mouse) if len(rec_by_mouse) else set()

def prioritized_order(common_set, priority_first):
    # case-insensitive matching but keep original casing from your data
    lower2orig = {g.lower(): g for g in common_set}
    out = []
    for p in priority_first:
        if p.lower() in lower2orig:
            out.append(lower2orig[p.lower()])
    # append the rest alphabetically (excluding those already added)
    rest = sorted([g for g in common_set if g not in out])
    return out + rest

# Put Spp1 first among ligands, Cd44 first among receptors
CANON_LIG_ORDER = prioritized_order(COMMON_LIGS, ["Spp1"])
CANON_REC_ORDER = prioritized_order(COMMON_RECS, ["Cd44"])

print(f"[Common sets] ligands={len(CANON_LIG_ORDER)}, receptors={len(CANON_REC_ORDER)}")

[Common sets] ligands=29, receptors=15


In [130]:
# ---- selection/visual knobs (tweak as needed) ----
TOP_N                 = 20      # top interactions per mouse by |Moran's I|
USE_ABS_MORANS_FOR_SELECTION = True

# link width from "mean" column only
UNITS_MIN, UNITS_MAX  = 0.6, 6.0         # min/max span units for link thickness

# circos aesthetics
CIRCOS_SPACE_DEG      = 8                # requested inter-sector gap (auto-safeguarded)
LABEL_SIZE            = 10
LABEL_R               = 105#118
TITLE_PREFIX          = "LIANA interactions per mouse · width=mean · color=Moran's I"

# --- global color normalization (for link colors) ---
GLOBAL_COLOR_NORM = True
GLOBAL_VMIN = min(df["morans"])
GLOBAL_VMAX = max(df["morans"])

print("Cell 1 ✓  (imports, paths, knobs)")

Cell 1 ✓  (imports, paths, knobs)


In [131]:
# Use python engine to auto-detect delimiter; if you KNOW it's tab, set sep="\t".


required = {
    "interaction", "ligand", "receptor",
    "ligand_means", "receptor_means",
    "morans", "morans_pvals",
    "mean", "std",
    "condition", "mouse"
}
missing = required - set(df.columns)
assert not missing, f"Missing columns in CSV: {missing}"

# coerce numeric columns robustly
num_cols = ["ligand_means", "receptor_means", "morans", "morans_pvals", "mean", "std"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# drop rows lacking essentials for plotting
df = df.dropna(subset=["ligand", "receptor", "morans", "mean"]).copy()

# normalize types
df["ligand"]   = df["ligand"].astype(str)
df["receptor"] = df["receptor"].astype(str)
df["mouse"]    = df["mouse"].astype(str)

print(f"Cell 2 ✓  Loaded {len(df):,} rows. "
      f"Mice detected: {sorted(df['mouse'].dropna().unique().tolist())}")

Cell 2 ✓  Loaded 413 rows. Mice detected: ['CyPSCA_1_1', 'NoTx_2_2', 'RTCyPSCA_1_4', 'RTCyPSCA_2_4']


In [132]:
# --- CELL 3: quick inventory by mouse ---
inv = (df.groupby(["mouse", "condition"], dropna=False)
         .size().reset_index(name="n_rows")
         .sort_values(["mouse", "condition"]))
display(inv.head(20))

print("Cell 3 ✓  Inventory shown (top rows).")


,mouse,condition,n_rows
0,CyPSCA_1_1,CyPSCA,107
1,NoTx_2_2,NoTx,94
2,RTCyPSCA_1_4,RTCyPSCA,68
3,RTCyPSCA_2_4,RTCyPSCA,144


Cell 3 ✓  Inventory shown (top rows).


In [135]:
pretty

'RTCyPSCA (No RT)'

In [147]:
from collections import defaultdict, OrderedDict

mice = sorted(df["mouse"].dropna().unique())
print(f"Found {len(mice)} mice: {mice}")

for mouse_id in mice:
    sub = df.query("mouse == @mouse_id").copy()
    if sub.empty:
        print(f"[skip] mouse={mouse_id}: no rows after filter.")
        continue

    # (optional) If your CSV has repeated rows per ligand/receptor within a mouse,
    # aggregate by mean to de-duplicate.
    group_cols = ["interaction", "ligand", "receptor"]
    agg = {
        "ligand_means": "mean",
        "receptor_means": "mean",
        "morans": "mean",
        "morans_pvals": "mean",
        "mean": "mean",  # <- width metric ONLY from 'mean'
    }
    g = sub.groupby(group_cols, as_index=False).agg(agg)

    # Select top by Moran's I magnitude (or signed, if you prefer)
    sel_score = g["morans"].abs() if USE_ABS_MORANS_FOR_SELECTION else g["morans"]
    top = (g.assign(sel_score=sel_score)
             .sort_values("sel_score", ascending=False)
             .head(TOP_N)
             .copy())

    if top.empty:
        print(f"[skip] mouse={mouse_id}: TOP_N selection is empty.")
        continue

    # ---- link widths from 'mean' only ----
    width_raw = top["mean"].clip(lower=0.0).to_numpy()  # ensure non-negative
    wmin, wmax = float(width_raw.min()), float(width_raw.max())
    if np.isfinite(wmin) and np.isfinite(wmax) and wmax > wmin:
        widths = np.interp(width_raw, (wmin, wmax), (UNITS_MIN, UNITS_MAX))
    else:
        widths = np.full_like(width_raw, (UNITS_MIN + UNITS_MAX) / 2.0, dtype=float)
    top["span_units"] = widths

    # ---- colors from Moran's I with global normalization ----
    if GLOBAL_COLOR_NORM:
        vmin, vmax = float(GLOBAL_VMIN), float(GLOBAL_VMAX)
    else:
        vmin = float(top["morans"].min())
        vmax = float(top["morans"].max())
    
    # build a safe normalizer; center at 0 if we straddle zero
    if vmin == vmax:
        norm = mcolors.Normalize(vmin=vmin - 1e-6, vmax=vmax + 1e-6)
    elif (vmin < 0.0) and (vmax > 0.0):
        norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    else:
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    
    # Matplotlib 3.7+ colormap API
    import matplotlib as mpl
    cmap = mpl.colormaps.get_cmap("coolwarm")

    top["color_rgba"] = [cmap(norm(v)) for v in top["morans"]]

    # ---- compute sector sizes (sum of link spans touching the sector) ----
    lig_unique = sorted(top["ligand"].unique())
    rec_unique = sorted(top["receptor"].unique())
    overlap = set(lig_unique).intersection(rec_unique)

    def name_with_side(g, side):
        # disambiguate if a gene appears on both sides
        return f"{g} ({side})" if g in overlap else g

    lig_sz = defaultdict(float)
    rec_sz = defaultdict(float)
    for _, row in top.iterrows():
        lig = name_with_side(row["ligand"], "L")
        rec = name_with_side(row["receptor"], "R")
        s   = float(row["span_units"])
        lig_sz[lig] += s
        rec_sz[rec] += s

    # ---- sector order: lock COMMON first in a canonical order; others random or by size ----
    if LOCK_COMMON_POSITIONS:
        # Locked (present this mouse & in COMMON)
        locked_ligs = [name_with_side(g, "L") for g in CANON_LIG_ORDER if name_with_side(g, "L") in lig_sz]
        locked_recs = [name_with_side(g, "R") for g in CANON_REC_ORDER if name_with_side(g, "R") in rec_sz]

        # Free (non-common) for this mouse
        free_ligs = list(set(lig_sz.keys()) - set(locked_ligs))
        free_recs = list(set(rec_sz.keys()) - set(locked_recs))

        if SHUFFLE_NONCOMMON:
            rng = np.random.default_rng((abs(hash(mouse_id)) ^ RNG_SEED_BASE) & 0xFFFFFFFF)
            rng.shuffle(free_ligs)
            rng.shuffle(free_recs)
        else:
            # size-descending as an alternative
            free_ligs = [k for k,_ in sorted(((k, lig_sz[k]) for k in free_ligs), key=lambda kv: kv[1], reverse=True)]
            free_recs = [k for k,_ in sorted(((k, rec_sz[k]) for k in free_recs), key=lambda kv: kv[1], reverse=True)]

        lig_order = locked_ligs + free_ligs
        rec_order = locked_recs + free_recs
    else:
        # original size-based ordering
        lig_order = [k for k,_ in sorted(lig_sz.items(), key=lambda kv: kv[1], reverse=True)]
        rec_order = [k for k,_ in sorted(rec_sz.items(), key=lambda kv: kv[1], reverse=True)]

    # Build sectors dict in the final order
    sectors = OrderedDict()
    for k in lig_order:
        sectors[k] = max(lig_sz[k], UNITS_MIN)  # guard tiny sizes
    for k in rec_order:
        sectors[k] = max(rec_sz[k], UNITS_MIN)

    # ---- safe inter-sector spacing ----
    n_sec = len(sectors)
    max_space = 360.0 / max(1, n_sec)     # theoretical average gap
    SAFE_SPACE_DEG = max(0.5, min(CIRCOS_SPACE_DEG, max_space - 0.5))

    circos = Circos(sectors, space=SAFE_SPACE_DEG)

    # labels (strip side markers in the text)
    for sector in circos.sectors:
        label = sector.name.replace(" (L)", "").replace(" (R)", "")
        sector.text(label, r=LABEL_R, size=LABEL_SIZE)

    # ---- add links (pack consecutively per sector) ----
    offsets = {s.name: 0.0 for s in circos.sectors}
    for _, row in top.iterrows():
        lig = name_with_side(row["ligand"], "L")
        rec = name_with_side(row["receptor"], "R")
        s   = float(row["span_units"])
        col = row["color_rgba"]

        x0, x1 = offsets[lig], offsets[lig] + s
        y0, y1 = offsets[rec], offsets[rec] + s

        circos.link((lig, x0, x1), (rec, y0, y1),
                    color=col, ec="none", alpha=0.90, direction=1)

        offsets[lig] = x1
        offsets[rec] = y1

    # ---- draw and save ----
    fig = circos.plotfig()
    fig.set_size_inches(13.5, 10.0)
    # leave room on right for colorbar
    circos.ax.set_position([0.06, 0.10, 0.78, 0.80])

    # try to show a single condition if unambiguous
    conds = sorted(sub["condition"].dropna().unique().tolist())
    cond_tag = conds[0] if len(conds) == 1 else "mixed"

    # Use pretty label if available; otherwise show mouse + condition
    pretty = label_map.get(mouse_id, None)
    if pretty is not None:
        title_str = f"{pretty}"
    else:
        title_str = f"{TITLE_PREFIX}  ·  mouse={mouse_id}  ·  condition={cond_tag}"

    fig.suptitle(title_str, fontsize=11.5, y=0.99)

    sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cax = fig.add_axes([0.78, 0.22, 0.025, 0.56])
    cbar = fig.colorbar(sm, cax=cax, orientation="vertical")
    cbar.set_label("Moran's I", fontsize=10)

     # Disable tight_layout to avoid warning with manually-added colorbar axes
    fig.set_tight_layout(False)

    base = OUT_DIR / f"circos_liana_mouse-{mouse_id}_top{TOP_N}"
    fig.savefig(base.with_suffix(".png"), dpi=300)
    fig.savefig(base.with_suffix(".pdf"), dpi=300)
    plt.close(fig)

    # optional: also write out the exact rows used for provenance
    top_out = OUT_DIR / f"circos_selection_mouse-{mouse_id}_top{TOP_N}.csv"
    top.to_csv(top_out, index=False)

    print(f"[done] mouse={mouse_id} → {base.with_suffix('.png').name}, "
          f"{base.with_suffix('.pdf').name}  (sectors={len(sectors)}, links={len(top)})")

print("Cell 4 ✓  All mice processed.")

Found 4 mice: ['CyPSCA_1_1', 'NoTx_2_2', 'RTCyPSCA_1_4', 'RTCyPSCA_2_4']
[done] mouse=CyPSCA_1_1 → circos_liana_mouse-CyPSCA_1_1_top20.png, circos_liana_mouse-CyPSCA_1_1_top20.pdf  (sectors=19, links=20)
[done] mouse=NoTx_2_2 → circos_liana_mouse-NoTx_2_2_top20.png, circos_liana_mouse-NoTx_2_2_top20.pdf  (sectors=21, links=20)
[done] mouse=RTCyPSCA_1_4 → circos_liana_mouse-RTCyPSCA_1_4_top20.png, circos_liana_mouse-RTCyPSCA_1_4_top20.pdf  (sectors=20, links=20)
[done] mouse=RTCyPSCA_2_4 → circos_liana_mouse-RTCyPSCA_2_4_top20.png, circos_liana_mouse-RTCyPSCA_2_4_top20.pdf  (sectors=18, links=20)
Cell 4 ✓  All mice processed.


In [134]:
# --- CELL 5: optional check of saved outputs ---
from itertools import islice

saved = sorted(OUT_DIR.glob("circos_liana_mouse-*_top*.png"))
print(f"Found {len(saved)} PNGs in {OUT_DIR}")
for p in islice(saved, 10):
    print(" •", p.name)

print("Cell 5 ✓  Listed a few saved figures.")

Found 4 PNGs in /mnt/c/Users/jonan/Documents/1Work/RoseLab/Spatial/CAR_T/figures/R01_figs/Circos
 • circos_liana_mouse-CyPSCA_1_1_top20.png
 • circos_liana_mouse-NoTx_2_2_top20.png
 • circos_liana_mouse-RTCyPSCA_1_4_top20.png
 • circos_liana_mouse-RTCyPSCA_2_4_top20.png
Cell 5 ✓  Listed a few saved figures.
